# Creación de datos sintéticos


El dataset de interacciones tiene datos limitados, y presentan poca variabilidad, lo que perjudica el proceso de aprendizaje de los agentes. Por esto añadimos nuevos datos para demostrar el potencial de los algoritmos.

## 1- Primero procesamos el dataset para coger solo la informacion que necesitamos y codificar variables

In [2]:
import pandas as pd
df_prep = pd.read_csv("sequences_df_prep_EN.csv")

Codificamos usuario

In [3]:
from sklearn.preprocessing import LabelEncoder

df = df_prep.copy()
le_user = LabelEncoder()
encoded_user = le_user.fit_transform(df_prep['user'])
df['encoded_user']=encoded_user

Codificamos mixture y container

In [4]:
le_mix = LabelEncoder()
encoded_mix = le_mix.fit_transform(df_prep['mixture'])
df['encoded_mixture']=encoded_mix

le_container = LabelEncoder()
encoded_container = le_container.fit_transform(df_prep['container'])
df['encoded_container']=encoded_container

Convertimos de epoch a hora decimal

In [5]:
hora = pd.to_datetime(df_prep['initepoch'], unit='ms')
df['hora_decimal'] = hora.dt.hour + hora.dt.minute / 60 + hora.dt.second / 3600

Cogemos los datos que necesitamos. Por ahora son Usuario, dia de la semana, hora , shift, mix elegido, additive elegido, container elegido.

In [6]:
df = df[['encoded_user','initdayofweek', 'hora_decimal', 'shift', 'encoded_mixture', 'additive','encoded_container']]

## 2- Pefiles de usuarios

Primero cramos usuarios falsos con determinados comportamientos especificos que ayudaran a saber si el modelo de verdad a aprendido a recomendar.   

Crearemos dos usuarios:   
    -Uno que a la mañana tomara Mix 2 y a la tarde Mix 8.  
    -Otro que desde el lunes hasta e miércoles tomara Mix 3 y los jueves y viernes Mix 14

Usuario que toma diferente mix mañana/tarde

In [7]:
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import Metadata

metadata = Metadata.detect_from_dataframe(data=df, table_name='interactions')

synthesizer = GaussianCopulaSynthesizer(metadata)
synthesizer.fit(df)

synthetic_data = synthesizer.sample(num_rows=100)

c:\Users\manex\miniconda3\envs\programacion\Lib\site-packages\sdv\single_table\base.py:123: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [8]:
synthetic_data['encoded_user'] = 27

synthetic_data.loc[synthetic_data['hora_decimal'] < 12.0, 'encoded_mixture'] = 2
synthetic_data.loc[synthetic_data['hora_decimal'] < 12.0, 'additive'] = 3

synthetic_data.loc[synthetic_data['hora_decimal'] > 12.0, 'encoded_mixture'] = 8
synthetic_data.loc[synthetic_data['hora_decimal'] > 12.0, 'additive'] = 0

synthetic_data['encoded_container'] = 1

Usuario que toma diferente mix segun el dia de la semana

In [9]:
synthetic_data_2 = synthesizer.sample(num_rows=100)
synthetic_data_2['encoded_user'] = 28

synthetic_data_2.loc[synthetic_data_2['initdayofweek'] <= 3 , 'encoded_mixture'] = 3
synthetic_data_2.loc[synthetic_data_2['initdayofweek'] > 3, 'encoded_mixture'] = 14
synthetic_data_2['additive'] = 0
synthetic_data_2['encoded_container'] = 1

Finalmente metemos usuarios que toman diferentes mix-es poco representados

In [10]:
synthetic_data_3 = synthesizer.sample(num_rows=100)
synthetic_data_3['encoded_user'] = 29
synthetic_data_3['encoded_mixture'] = 6
synthetic_data_3['additive'] = 0
synthetic_data_3['encoded_container'] = 0

synthetic_data_4 = synthesizer.sample(num_rows=100)
synthetic_data_4['encoded_user'] = 30
synthetic_data_4['encoded_mixture'] = 7
synthetic_data_4['additive'] = 3
synthetic_data_4['encoded_container'] = 1

Los añadimos al dataframe, pero creamos nueva variable para saber si es sintetico o no

In [11]:
df = pd.concat([df, synthetic_data, synthetic_data_2, synthetic_data_3, synthetic_data_4], ignore_index=True)
df['synthetic'] = df['encoded_user'].isin([27, 28, 29, 30])

Y reordenamos las filas

In [12]:
df = df.sample(frac=1).reset_index(drop=True)

Guardamos el dataset

In [13]:
df.to_csv("cooked_df.csv", index=False)

Dividimos en entrenamiento/ test y los guardamos

In [ ]:
from sklearn.model_selection import train_test_split

cooked_train, cooked_test = train_test_split(df, test_size=0.2, random_state=42)

cooked_train.to_csv("cooked_train.csv", index=False)
cooked_test.to_csv("cooked_test.csv", index=False)

: 